Explore extracting data from Kaggle. Use Kaggle hub

In [ ]:
#Install the kagglehub package with pip:
%pip install kagglehub

In [12]:
# Load environment (only for notebook not for docker)
import os
from dotenv import load_dotenv

load_dotenv("../.env")


GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID")
GCS_BUCKET_NAME = os.getenv("GCS_BUCKET_NAME")
GCS_DATASET_ID = os.getenv("GCS_DATASET_ID")


print(GCP_PROJECT_ID)
print(GCS_BUCKET_NAME)
print(GCS_DATASET_ID)

my-project-bigdata-505312
olistcontainer
olist_raw


In [14]:
import kagglehub
from google.cloud import storage
from dotenv import load_dotenv
import os

# Load environment (only for notebook not for docker)
load_dotenv("../.env")


GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID")
GCS_BUCKET_NAME = os.getenv("GCS_BUCKET_NAME")
GCS_DATASET_ID = os.getenv("GCS_DATASET_ID")


print(GCP_PROJECT_ID)
print(GCS_BUCKET_NAME)
print(GCS_DATASET_ID)

ModuleNotFoundError: No module named 'google.cloud'

In [27]:
%%writefile ../src/extract/kaggle_extract.py
# you could use kagglehub to download the dataset and then the Google Cloud Storage Python library to upload it.
import os

import kagglehub
from google.cloud import storage


def extract_kaggle_to_gcs():

    GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]
    GCS_BUCKET_NAME = os.environ["GCS_BUCKET_NAME"]

    print("GCP Project:", GCP_PROJECT_ID)
    print("GCS Bucket:", GCS_BUCKET_NAME)

    # 1. Download dataset from Kaggle
    path = kagglehub.dataset_download(
        "olistbr/brazilian-ecommerce"
    )

    print("Downloaded to:", path)

    # 2. Connect to GCS
    client = storage.Client(project=GCP_PROJECT_ID)

    bucket = client.bucket(GCS_BUCKET_NAME)

    print("Connected to bucket:", bucket.name)

    # 3. Upload files
    for filename in os.listdir(path):

        local_file = os.path.join(path, filename)

        if os.path.isfile(local_file):

            blob = bucket.blob(
                f"raw/kaggle/{filename}"
            )

            blob.upload_from_filename(local_file)

            print(f"Uploaded: {filename}")

Overwriting ../src/extract/kaggle_extract.py


# test extract call in docker 
docker compose exec app python -c "from src.extract.kaggle_extract import extract_kaggle_to_gcs; extract_kaggle_to_gcs()"

In [ ]:
import os
print(os.environ["PATH"])

!which docker
'/Users/priya/Documents/DS6/Big data engineering/Project/Repo/module-2-project/src/extract/kaggle_extract.py'

# Run extracter
!../docker compose --env-file .env run --rm app python ../src/extract/kaggle_extract.py

/Users/priya/miniconda3/envs/elt/bin:/Users/priya/miniconda3/condabin:/usr/local/bin:/System/Cryptexes/App/usr/bin:/usr/bin:/bin:/usr/sbin:/sbin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/local/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/appleinternal/bin:/pkg/env/global/bin:/Library/Apple/usr/bin:/opt/homebrew/bin
docker not found
zsh:1: no such file or directory: ../docker


In [ ]:
%pip install google-cloud-bigquery

In [30]:
%%writefile ../src/loader/kaggle_load.py
# Create big querry data set 
import os

from google.cloud import bigquery


def load_gcs_to_bigquery():

    GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]
    GCS_BUCKET_NAME = os.environ["GCS_BUCKET_NAME"]
    GCS_DATASET_ID = os.environ["GCS_DATASET_ID"]

    print("GCP Project:", GCP_PROJECT_ID)
    print("GCS Bucket:", GCS_BUCKET_NAME)
    print("BigQuery Dataset:", GCS_DATASET_ID)

    client = bigquery.Client(project=GCP_PROJECT_ID)

    print("Connected to BigQuery!")

    # Create BigQuery dataset
    dataset_ref = f"{GCP_PROJECT_ID}.{GCS_DATASET_ID}"

    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = "asia-southeast1"

    client.create_dataset(dataset, exists_ok=True)

    print(f"BigQuery dataset ready: {dataset_ref}")

    files = [
        "olist_sellers_dataset.csv",
        "product_category_name_translation.csv",
        "olist_orders_dataset.csv",
        "olist_order_items_dataset.csv",
        "olist_customers_dataset.csv",
        "olist_geolocation_dataset.csv",
        "olist_order_payments_dataset.csv",
        "olist_order_reviews_dataset.csv",
        "olist_products_dataset.csv"
    ]

    for filename in files:

        table_name = filename.replace(".csv", "")
        table_name = table_name.replace("_dataset", "")

        table_id = (
            f"{GCP_PROJECT_ID}."
            f"{GCS_DATASET_ID}."
            f"{table_name}"
        )

        uri = (
            f"gs://{GCS_BUCKET_NAME}/"
            f"raw/kaggle/{filename}"
        )

        job_config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.CSV,
            skip_leading_rows=1,
            autodetect=True,
            allow_quoted_newlines=True,
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
        )

        print(f"Loading {filename} → {table_id}")

        load_job = client.load_table_from_uri(
            uri,
            table_id,
            job_config=job_config
        )

        load_job.result()

        print(f"✓ Loaded: {table_name}")

Overwriting ../src/loader/kaggle_load.py


In [ ]:
# Run loader
!docker compose --env-file .env run --rm app python ../src/loader/kaggle_load.py

#test loader in docker 
docker compose exec app python -c "from src.loader.kaggle_load import load_gcs_to_bigquery; load_gcs_to_bigquery()"

In [ ]:
# invoke docker and run any commands inside eg. dbt
docker compose exec app bash

# run below command in app bash to move tables to staging  in big querry
root@7725bfb64667:/module-2-project/dbt/olist_dbt# python -c "from google.cloud import bigquery; c=bigquery.Client(); tables=c.list_tables('my-project-bigdata-505312.olist_raw'); print('\n'.join(t.table_id for t in tables))"
olist_customers
olist_geolocation
olist_order_items
olist_order_payments
olist_order_reviews
olist_orders
olist_products
olist_sellers
product_category_name_translation

# move data to staging area

root@7725bfb64667:/module-2-project/dbt/olist_dbt# mkdir -p models/staging
root@7725bfb64667:/module-2-project/dbt/olist_dbt# cat > models/staging/sources.yml <<'EOF'
# yml version 
version: 2  

# indicate it as source and -name is the source group name in dbt, Schema is the dataset name
sources:
  - name: olist
    database: my-project-bigdata-505312
    schema: olist_raw

    tables:
      - name: olist_customers
      - name: olist_geolocation
      - name: olist_order_items
      - name: olist_order_payments
      - name: olist_order_reviews
      - name: olist_orders
      - name: olist_products
      - name: olist_sellers
      - name: product_category_name_translation
EOF

# Ask dbt to parse the project
# Now run:
# dbt parse

# This is an important concept.
# dbt first needs to read and understand your project.
# dbt parse is a safe first validation step.

# Raw data
#    ↓
# dbt parse          ← "Can dbt understand my project?"
#    ↓
# dbt source         ← "Where is my raw data?"
#    ↓
# Staging models     ← "Now clean and standardize the data"
#    ↓
# Analytics models   ← "Transform it for business use" -->

- go to docker 
(base) priya@Sivakumars-MacBook-Pro module-2-project % docker compose exec app bash

- Navigate to Olist_dbt and run dbt parse
# So the rule to remember is:
# Run dbt commands from the directory containing dbt_project.yml.
root@7725bfb64667:/module-2-project# cd dbt/olist_dbt/
root@7725bfb64667:/module-2-project/dbt/olist_dbt# dbt parse

- Verify if dbt can recognize source 
dbt ls --resource-type source

# Next step is creating analytical models 
- Run this in docker to know what is in the table, columns, data type, where we need cleaning etc.
python -c "from google.cloud import bigquery; c=bigquery.Client(); t=c.get_table('my-project-bigdata-505312.olist_raw.olist_orders'); print('\n'.join(f'{x.name}: {x.field_type}' for x in t.schema))"
- Run below in docker to Create stg_orders.sql - this creates model 
cat > models/staging/stg_orders.sql <<'EOF'
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
FROM {{ source('olist', 'olist_orders') }}
EOF
- Verify
cat models/staging/stg_orders.sql
- compile (dbt takes dbt sql and converts it into actual SQL that BigQuery understands)
# Your dbt SQL
#      ↓
# dbt compile
#      ↓
# Actual BigQuery SQL

dbt compile --select stg_orders

- Run (asking dbt to create the transformed object in BigQuery, output will go into target data set)
dbt run --select stg_orders  

- Verify (querry the view to say data looks correct) below command in docker
python -c "from google.cloud import bigquery; c=bigquery.Client(); q='SELECT * FROM \`my-project-bigdata-505312.olist_analytics.stg_orders\` LIMIT 5'; rows=c.query(q).result(); [print(dict(r)) for r in rows]"

# next step
Run the four profiling checks we discussed:
Row count
Order-status distribution
NULL counts
Duplicate order_id

In [34]:
%%writefile ../src/profiling/profile_orders.py 

from google.cloud import bigquery
import os


def profile_orders():

    GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]
    GCS_DATASET_ID = os.environ["GCS_DATASET_ID"]

    # ---------------------------------------------------------
    # Configuration
    # ---------------------------------------------------------

    TABLE = "olist_orders"

    TABLE_ID = f"{GCP_PROJECT_ID}.{GCS_DATASET_ID}.{TABLE}"

    # ---------------------------------------------------------
    # BigQuery connection
    # ---------------------------------------------------------

    client = bigquery.Client(project=GCP_PROJECT_ID)

    print("=" * 70)
    print("DATA PROFILING REPORT")
    print("=" * 70)
    print(f"Table: {TABLE_ID}")
    print()

    # ---------------------------------------------------------
    # 1. Row count
    # ---------------------------------------------------------

    print("1. ROW COUNT")
    print("-" * 70)

    query = f"""
    SELECT COUNT(*) AS row_count
    FROM `{TABLE_ID}`
    """

    result = list(client.query(query).result())
    row_count = result[0]["row_count"]

    print(f"Total rows: {row_count:,}")
    print()

    # ---------------------------------------------------------
    # 2. Order status distribution
    # ---------------------------------------------------------

    print("2. ORDER STATUS DISTRIBUTION")
    print("-" * 70)

    query = f"""
    SELECT
        order_status,
        COUNT(*) AS count
    FROM `{TABLE_ID}`
    GROUP BY order_status
    ORDER BY count DESC
    """

    results = client.query(query).result()

    for row in results:
        print(f"{row['order_status']}: {row['count']:,}")

    print()

    # ---------------------------------------------------------
    # 3. NULL value analysis
    # ---------------------------------------------------------

    print("3. NULL VALUE ANALYSIS")
    print("-" * 70)

    query = f"""
    SELECT
        COUNTIF(order_id IS NULL) AS order_id_nulls,
        COUNTIF(customer_id IS NULL) AS customer_id_nulls,
        COUNTIF(order_status IS NULL) AS status_nulls,
        COUNTIF(order_purchase_timestamp IS NULL) AS purchase_nulls,
        COUNTIF(order_approved_at IS NULL) AS approved_nulls,
        COUNTIF(order_delivered_carrier_date IS NULL) AS carrier_nulls,
        COUNTIF(order_delivered_customer_date IS NULL) AS delivered_nulls,
        COUNTIF(order_estimated_delivery_date IS NULL) AS estimated_nulls
    FROM `{TABLE_ID}`
    """

    row = list(client.query(query).result())[0]

    null_columns = {
        "order_id": row["order_id_nulls"],
        "customer_id": row["customer_id_nulls"],
        "order_status": row["status_nulls"],
        "order_purchase_timestamp": row["purchase_nulls"],
        "order_approved_at": row["approved_nulls"],
        "order_delivered_carrier_date": row["carrier_nulls"],
        "order_delivered_customer_date": row["delivered_nulls"],
        "order_estimated_delivery_date": row["estimated_nulls"],
    }

    for column, count in null_columns.items():
        percentage = (count / row_count * 100) if row_count else 0
        print(f"{column:40} {count:8,} ({percentage:6.2f}%)")

    print()

    # ---------------------------------------------------------
    # 4. Duplicate order IDs
    # ---------------------------------------------------------

    print("4. DUPLICATE ORDER IDs")
    print("-" * 70)

    query = f"""
    SELECT
        order_id,
        COUNT(*) AS count
    FROM `{TABLE_ID}`
    GROUP BY order_id
    HAVING COUNT(*) > 1
    ORDER BY count DESC
    LIMIT 20
    """

    results = list(client.query(query).result())

    if not results:
        print("No duplicate order_ids found.")
    else:
        print("Duplicate order_ids detected.")
        print("Showing up to 20 examples:")
        print()

        for row in results:
            print(
                f"order_id: {row['order_id']} | "
                f"count: {row['count']}"
            )

    print()

    # ---------------------------------------------------------
    # 5. Purchase date range
    # ---------------------------------------------------------

    print("5. PURCHASE DATE RANGE")
    print("-" * 70)

    query = f"""
    SELECT
        MIN(order_purchase_timestamp) AS earliest_order,
        MAX(order_purchase_timestamp) AS latest_order
    FROM `{TABLE_ID}`
    """

    row = list(client.query(query).result())[0]

    print(f"Earliest order: {row['earliest_order']}")
    print(f"Latest order:   {row['latest_order']}")
    print()

    # ---------------------------------------------------------
    # 6. Invalid delivery dates
    # ---------------------------------------------------------

    print("6. INVALID DELIVERY DATES")
    print("-" * 70)

    query = f"""
    SELECT COUNT(*) AS invalid_delivery_dates
    FROM `{TABLE_ID}`
    WHERE order_delivered_customer_date < order_purchase_timestamp
    """

    row = list(client.query(query).result())[0]

    print(
        f"Orders delivered before purchase: "
        f"{row['invalid_delivery_dates']:,}"
    )
    print()

    # ---------------------------------------------------------
    # 7. Invalid estimated delivery dates
    # ---------------------------------------------------------

    print("7. INVALID ESTIMATED DELIVERY DATES")
    print("-" * 70)

    query = f"""
    SELECT COUNT(*) AS invalid_count
    FROM `{TABLE_ID}`
    WHERE DATE(order_estimated_delivery_date)
          < DATE(order_purchase_timestamp)
    """

    row = list(client.query(query).result())[0]

    print(
        f"Orders with estimated delivery before purchase: "
        f"{row['invalid_count']}"
    )

    # ---------------------------------------------------------
    # Profiling complete
    # ---------------------------------------------------------

    print("=" * 70)
    print("PROFILING COMPLETE")
    print("=" * 70)

Overwriting ../src/profiling/profile_orders.py


# Run above python script in docker 
docker exec -it module-2-project python /module-2-project/src/profiling/profile_orders.py

# Test profiling output
docker compose exec app python -c "from src.profiling.profile_orders import profile_orders; profile_orders()"

# create staging
dbt/
└── olist_dbt/
    ├── dbt_project.yml
    ├── README.md
    └── models/
        └── staging/

updated sbt_project.yml

# add staging and related sources
dbt source name
      │
      │ source('olist', 'olist_orders')
      ▼
BigQuery
my-project-bigdata-505312
        │
        └── olist_raw
              │
              └── olist_orders

# on dbt run 
BigQuery RAW
olist_raw.olist_orders
        │
        │
        ▼
      dbt
        │
        ▼
olist_analytics              

In [24]:
%%writefile ../dbt/Olist_dbt/models/staging/stg_orders.sql 

SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
FROM {{ source('olist', 'olist_orders') }}

Overwriting ../dbt/Olist_dbt/models/staging/stg_orders.sql


In [ ]:

# staging

dbt run --select stg_orders

# building analytical model 
mothly order volume is the analytical one 

olist_raw
    │
    ▼
stg_orders                 ← VIEW
    │
    ▼
monthly_order_volume      ← TABLE
    │
    ▼
Streamlit

- create marts directory
- create sql querry monthy order volume 
- configure mart table in dbt_project.yml file 


In [23]:
%%writefile ../dbt/Olist_dbt/models/marts/monthly_order_volume.sql

{{ config(materialized='table') }}

SELECT
    DATE_TRUNC(
        DATE(order_purchase_timestamp),
        MONTH
    ) AS order_month,

    COUNT(DISTINCT order_id) AS order_count

FROM {{ ref('stg_orders') }}

GROUP BY order_month
ORDER BY order_month

Overwriting ../dbt/Olist_dbt/models/marts/monthly_order_volume.sql


In [ ]:
#dbt run

dbt run --select monthly_order_volume

# BigQuery
# └── olist_analytics
#     └── monthly_order_volume

In [ ]:
#  look at actual analytical data 
# preview 5 
dbt show --select monthly_order_volume
#view full table 
dbt show --select monthly_order_volume --limit 30

# Streamlit
- Create streamlit folder 
- Create app.py 
- Modify compose.yml to add streamlit
- rebuild docker 
docker compose up -d --build
- run streamlit in docker env
docker compose exec app ls -la /module-2-project/streamlit

Step 1  → Streamlit runs
            ↓
Step 2  → Connect to BigQuery
            ↓
Step 3  → Read monthly_order_volume
            ↓
Step 4  → Display the 25 months
            ↓
Step 5  → Add the chart
            ↓
Step 6  → Add business-event analysis

- add streamlit in docker requirements.txt
- rebuild docker 
docker compose up -d --build
- run streamlit
docker compose exec app streamlit run /module-2-project/streamlit/app.py --server.address=0.0.0.0
- Since docker is used streamlit in docker port need to be mapped to mac port before we can see 
- add ports in compose.yml

In [21]:
%%writefile ../streamlit/app.py

import os

import pandas as pd
import streamlit as st
from google.cloud import bigquery


# --------------------------------------------------
# Streamlit page configuration
# --------------------------------------------------

st.set_page_config(
    page_title="Olist Analytics",
    page_icon="📊",
    layout="wide"
)

st.title("📊 Olist Analytics Dashboard")
st.write("Historical monthly order volume")


# --------------------------------------------------
# BigQuery configuration
# --------------------------------------------------

PROJECT_ID = os.environ["GCP_PROJECT_ID"]

TABLE_ID = (
    f"{PROJECT_ID}."
    "olist_analytics."
    "monthly_order_volume"
)


# --------------------------------------------------
# Query BigQuery
# --------------------------------------------------

@st.cache_data
def load_data():

    client = bigquery.Client(project=PROJECT_ID)

    query = f"""
        SELECT
            order_month,
            order_count
        FROM `{TABLE_ID}`
        ORDER BY order_month
    """

    df = client.query(query).to_dataframe()

    return df


# --------------------------------------------------
# Load data
# --------------------------------------------------

df = load_data()


# --------------------------------------------------
# Display data
# --------------------------------------------------

st.subheader("Monthly Order Volume")

st.dataframe(
    df,
    use_container_width=True
)


# --------------------------------------------------
# Display chart
# --------------------------------------------------

st.subheader("Order Volume Trend")

chart_df = df.set_index("order_month")

st.line_chart(
    chart_df["order_count"]
)


# --------------------------------------------------
# Data interpretation
# --------------------------------------------------

st.info(
    "Note: September and October 2018 contain incomplete "
    "data because the source dataset ends in October 2018."
)



Overwriting ../streamlit/app.py


# Next step is to introduce dagster to do the orchestration
              ┌─────────────────────┐
              │       DAGSTER       │
              │                     │
              │  controls order     │
              │  dependencies       │
              │  retries            │
              │  scheduling         │
              │  monitoring         │
              │  data lineage       │
              └─────────┬───────────┘
                        │
       ┌────────────────┼────────────────┐
       ▼                ▼                ▼
    Extract           Profile           dbt
       │                │                │
       └──────────────► ┴ ◄──────────────┘
                        │
                        ▼
                   Analytics
                        │
                        ▼
                   Streamlit

# 1. Create dagster frame work first 
Module-2-Project/
│
├── src/
│   ├── extract/
│   ├── load/
│   ├── profiling/
│   └── ...
│
├── dbt/
│   └── olist_dbt/
│
├── streamlit/
│   └── app.py
│
└── dagster/
    ├── definitions.py
    └── ...

- Check dagster installation
root@a3af08d8685c:/module-2-project# dagster --version
bash: dagster: command not found
- add dagster to requirements.txt
dagster
dagster-webserver
- Rebuild docker 
docker compose down
docker compose up -d --build
- verify dagster
docker compose exec app dagster --version
docker compose exec app python -c "import dagster; print(dagster.__version__)"
- add dagster in docker compose.yml
./dagster:/module-2-project/dagster
- create dagster directory 
mkdir -p dagster
- Rebuild docker
docker compose down
docker compose up -d
- verify
docker compose exec app ls -la /module-2-project/dagster
- Create a file 'definitions.py' in dagster like below




In [44]:
%%writefile ../dagster/definitions.py
# Create Dagster assets in this file

from pathlib import Path

from dagster import asset, Definitions, ScheduleDefinition, define_asset_job, AssetSelection
from dagster_dbt import DbtCliResource, DagsterDbtTranslator, dbt_assets

from src.extract.kaggle_extract import extract_kaggle_to_gcs
from src.loader.kaggle_load import load_gcs_to_bigquery
from src.profiling.profile_orders import profile_orders


# ---------------------------------------------------------
# Existing pipeline assets
# ---------------------------------------------------------

@asset
def kaggle_to_gcs():
    extract_kaggle_to_gcs()


@asset(deps=[kaggle_to_gcs])
def gcs_to_bigquery_raw():
    load_gcs_to_bigquery()


@asset(deps=[gcs_to_bigquery_raw])
def profile_orders_asset():
    profile_orders()


# ---------------------------------------------------------
# DBT configuration
# ---------------------------------------------------------

dbt_project_dir = Path("/module-2-project/dbt/olist_dbt")

dbt = DbtCliResource(
    project_dir=dbt_project_dir,
    profiles_dir="/module-2-project/dbt/profiles",
)


# ---------------------------------------------------------
# DBT assets
# ---------------------------------------------------------
class CustomDbtTranslator(DagsterDbtTranslator):

    def get_asset_spec(self, manifest, unique_id, project):

        spec = super().get_asset_spec(
            manifest,
            unique_id,
            project
        )

        if spec.key.to_user_string() == "stg_orders":
            spec = spec.merge_attributes(
                deps=[gcs_to_bigquery_raw]
            )

        return spec

@dbt_assets(
    manifest=dbt_project_dir / "target" / "manifest.json",
    dagster_dbt_translator=CustomDbtTranslator(),
)
def olist_dbt_assets(context, dbt: DbtCliResource):
    yield from dbt.cli(["build"], context=context).stream()

# ---------------------------------------------------------
# Dagster job and schedule
# ---------------------------------------------------------

daily_pipeline_job = define_asset_job(
    name="daily_olist_pipeline",
    selection=AssetSelection.all(),
)

daily_olist_schedule = ScheduleDefinition(
    job=daily_pipeline_job,
    cron_schedule="0 2 * * *",
    execution_timezone="Asia/Singapore",
)

# ---------------------------------------------------------
# Dagster definitions
# ---------------------------------------------------------

defs = Definitions(
    assets=[
        kaggle_to_gcs,
        gcs_to_bigquery_raw,
        profile_orders_asset,
        olist_dbt_assets,
    ],
    resources={
        "dbt": dbt,
    },
    jobs=[
        daily_pipeline_job,
    ],
    schedules=[
        daily_olist_schedule,
    ],
)

Overwriting ../dagster/definitions.py


# Run above file in docker environment 
root@f476002c226a:/module-2-project/dagster# python definitions.py
definitions

# Run dagster assets - just to check if dagster works
dagster asset list -f definitions.py
dagster asset materialize -f definitions.py

# Improvise dagster to add all the pipeline operations
- add Kaggle to GCS (test seperately first before adding here)

- test run and materialise
root@f476002c226a:/module-2-project/dagster# dagster asset list -f definitions.py -d /module-2-project
kaggle_to_gcs

- Materialise
dagster asset materialize -f definitions.py -d /module-2-project --select kaggle_to_gcs
- now done with below 
Kaggle
   │
   ▼
Dagster
   │
   ▼
kaggle_to_gcs
   │
   ▼
GCS

# Build next depenency - gcs raw to big querry

        kaggle_to_gcs
              │
              │ dependency
              ▼
     gcs_to_bigquery_raw
- add load gcs to big query function in definitions.py and add dependency
@asset(deps=[kaggle_to_gcs])
- verify if dependency added 
dagster asset list -f definitions.py -d /module-2-project
- Run the actual orchestration cmd and verify the sequence is correct 
dagster asset materialize -f definitions.py -d /module-2-project --select "*"

# Build next dependency - big querry raw to profiling
gcs_to_bigquery_raw
          │
          ▼
   profile_orders
- update definitions.py in dagster folder 
- verify if it works 
docker compose exec app bash -c "cd /module-2-project/dagster && dagster asset list -f definitions.py -d /module-2-project"
docker compose exec app bash -c "cd /module-2-project/dagster && dagster asset materialize -f definitions.py -d /module-2-project --select profile_orders_asset"

# Linking dagster and dbt
- verify dagster and dbt integration is installed 
docker compose exec app bash -c "python -c \"import dagster_dbt; print('dagster-dbt is installed')\""
- if dagster-dbt is not installed add it in requriements.txt of docker
- rebuild docker image 
docker compose build app 
docker compose up -d --force-recreate
- Verify dagster-dbt
docker compose exec app python -c "import dagster_dbt; print('dagster-dbt is installed')"
- update definitions.py to include dagster-dbt
gcs_to_bigquery_raw
        ↓
profile_orders_asset
        ↓
   dbt assets
      ↓
 stg_orders
      ↓
monthly_order_volume
Dagster will not start the dbt portion until profiling has completed successfully.

- profiling cant be linked as dependency to dbt because the flow would be like below and profiling will not create any table output
                    BigQuery RAW
                         │
                  olist_orders
                    /       \
                   /         \
                  ▼           ▼
          profile_orders    dbt
             .py              │
              │               ▼
              │          stg_orders
              │               │
              │               ▼
              │       monthly_order_volume
              │
              ▼
          Data quality
            report

- Due to above reasons materialize only big querry and dbt first
docker compose exec app bash -c "cd /module-2-project/dagster && dagster asset materialize -f definitions.py -d /module-2-project --select '+monthly_order_volume'"
- Materialise complete pipeline 
docker compose exec app bash -c "cd /module-2-project/dagster && dagster asset materialize -f definitions.py -d /module-2-project --select '*'"
- add additional changes to bring in the bigquery_raw -> profile asset -> dbt asset linkage at present dbt and dagster are linked. Update dbt-generated AssetSpecs that they depend on gcs_to_bigquery_raw
- add changes to definitions.py the dependencies and dbt translator dbt input is olist raw we need to say that dbt shoould use this olist raw after gcs to big query is executed. Run below to verify dependency
docker compose exec app python -c "import runpy; d=runpy.run_path('/module-2-project/dagster/definitions.py'); [(print('ASSET:', s.key, 'DEPS:', s.deps)) for s in d['olist_dbt_assets'].specs]"
- Full pipeline is set now. Dagster execution order can be verified using asset graph in below cmd
docker compose exec app bash -c "cd /module-2-project/dagster && dagster asset list -f definitions.py -d /module-2-project"
- command to materialise dagter - dbt flow without kaggle and big querry
docker compose exec app bash -c "cd /module-2-project/dagster && dagster asset materialize -f definitions.py -d /module-2-project --select '+monthly_order_volume'"
- Command to materialise complete dagster flow from kaggle -> big querry -> dagster
docker compose exec app bash -c "cd /module-2-project/dagster && dagster asset materialize -f definitions.py -d /module-2-project --select '*'"
                 ┌───────────────┐
                 │    Kaggle     │
                 └───────┬───────┘
                         ↓
                  kaggle_to_gcs
                         ↓
                 ┌───────────────┐
                 │      GCS      │
                 └───────┬───────┘
                         ↓
              gcs_to_bigquery_raw
                         ↓
                 ┌───────┴────────┐
                 ↓                ↓
             Profiling           dbt
                                  ↓
                            stg_orders
                                  ↓
                       monthly_order_volume
                                  ↓
                             Streamlit

<!-- Next step: Scheduling
Now we can move to the part you originally wanted: automatically running the whole pipeline with Dagster.
We'll do this carefully:
Step 1: create a Dagster schedule
Step 2: verify Dagster recognizes it
Step 3: start the Dagster webserver
Step 4: view the schedule in the UI
Step 5: enable the schedule
Step 6: later decide the frequency (daily/weekly/etc.) -->

# Step 1: create a Dagster schedule
- add a dagster job and schedule 
add pipeline job to definitions.py "daily_pipeline job" and link job in Definitions
daily_olist_schedule
        │
        │ every day at 2:00 AM Singapore time
        ▼
daily_pipeline_job
        │
        ▼
ALL Dagster assets
        │
        ├── kaggle_to_gcs
        │
        └── gcs_to_bigquery_raw
                  │
                  ├── profile_orders_asset
                  │
                  └── dbt
                       ├── stg_orders
                       └── monthly_order_volume

# Step 2. verify if dagster can recognize this change
docker compose exec app bash -c "cd /module-2-project/dagster && dagster schedule list -f definitions.py -d /module-2-project"
- Above command needs dagster home to be set 
mkdir -p dagster_home
ls -ld dagster_home
- add dagster home in docker compose under volumes
- add dagster home in the environment yml
- recreate docker container 
docker compose up -d --force-recreate
- verify dagster home 
docker compose exec app bash -c 'echo $DAGSTER_HOME'
- use first command to verify if dagster can recognize now
- dagster can recognize dagster home and look for schedule now create schedule in dagster.yml file inside dagster (out put shows dagster scheduled using cron jobs by default 2am daily)
docker compose exec app bash -c "cd /module-2-project/dagster && dagster schedule list -f definitions.py -d /module-2-project"


# Step 3: start the Dagster webserver
- Expose dagster port 3000 in docker to start the Dagster UI
add port in compose yml
- Recreate docker container 
docker compose up -d --force-recreate
- verify port 
docker compose ps
- start dagster webserver
docker compose exec app bash -c "cd /module-2-project/dagster && dagster-webserver -h 0.0.0.0 -p 3000 -f definitions.py -d /module-2-project"

# Step 4: view the schedule in the UI
- Once above comand runs open below link in the browser - in the web page left hand side automation look for daily olist job scheduled, turn on the enable button
http://localhost:3000 
             Dagster Scheduler
                    │
             Every day 2 AM
                    │
                    ▼
              Kaggle → GCS
                    │
                    ▼
           GCS → BigQuery RAW
                    │
              ┌─────┴─────┐
              ▼           ▼
          Profiling      dbt
                          │
                          ▼
                 BigQuery Analytics
                          │
                          ▼
                     Streamlit
# Step 5: enable the schedule and test if it runs
- Next step is to test - run the scheduler daemon (open a 2nd mac terminal and run this command and keep that terminal open too)
docker compose exec app bash -c "cd /module-2-project/dagster && dagster-daemon run -f definitions.py -d /module-2-project"
The key line is:
Instance is configured with the following daemons:
['AssetDaemon', 'BackfillDaemon', 'FreshnessDaemon',
 'QueuedRunCoordinatorDaemon', 'SchedulerDaemon', 'SensorDaemon']
Most importantly, SchedulerDaemon is now running.